# A callable stepped-coupon bond

We value an amortizing six-year USD note whose first coupon is fixed and whose
later coupons float with a cap and floor. The issuer can call during specified
windows. The exercise right makes the value depend on future rates, so we
calibrate a short-rate model and compare two backward-pricing engines.

This notebook concentrates on the contract and its valuation. File loading is
in [the local loading helper](market_data.py), numerical/model settings are in
[data/bond_tree.json](data/bond_tree.json) and
[data/bond_fourier.json](data/bond_fourier.json), and script construction has
moved to [its own walkthrough](../scripting/scripted_cashflows.ipynb). See [setup](../README.md).

In [16]:
import pandas as pd
import quantsupport as qs
from market_data import load_market, load_pricers

market = load_market()
ref = market.quotes.reference_date
tree_configuration = load_pricers("bond_tree.json")
fourier_configuration = load_pricers("bond_fourier.json")
print(
    "Market reference date:",
    ref,
)

Market reference date: 2026-02-24


## Write the contract in financial terms

`Coupon.fixed` and `Coupon.floating` describe coupon economics; `CallScheduleEntry`
describes issuer exercise. Typed dates and enums keep those decisions visible.
The coupon and outstanding-principal lists are **positional by payment period**:
the last element repeats for all later periods. Thus the 4.5% coupon applies
only to the first half-year, not the first year.

Outstanding principal is 100% in period one, 90% in period two, then 75% through
maturity. Reductions are paid at period ends and the remaining principal is
repaid at maturity. Fixed accrual uses Actual/365 here; the floating coupon
uses SOFR's Actual/360 convention. The cap and floor bound the all-in floating
rate including its 60 bp spread.

In [ ]:
bond = qs.SteppedCouponBond(
    identifier="USD-STEPPED-CALL-6Y",
    start_date=ref,
    maturity_date=ref + "6Y",
    notional=1_000_000,
    units=100.0,
    currency=qs.Currency.USD,
    payment_frequency=qs.Frequency.Semiannual,
    coupon_day_counter=qs.DayCounter.Actual365,
    coupons=[
        qs.Coupon.fixed(0.045),
        qs.Coupon.floating(
            qs.MarketIndex.SOFR,
            spread=0.006,
            floor=0.0,
            cap=0.06,
        ),
    ],
    notional_percentages=[1.0, 0.90, 0.75],
    call_schedule=[
        qs.CallScheduleEntry.window(
            ref + "3Y",
            ref + "3Y3M",
            101.0,
        ),
        qs.CallScheduleEntry.window(
            ref + "4Y",
            ref + "4Y3M",
            100.0,
        ),
    ],
)
bond

SteppedCouponBond(identifier='USD-STEPPED-CALL-6Y', start_date='2026-02-24', maturity_date='2032-02-24', notional=1000000, coupons=2, notionals=3, calls=2)

Each call window includes both boundaries. Strikes are quoted per 100 of
**outstanding** principal. On an exercise date, contractual coupon and
amortization due that day are paid; the strike caps the remaining bond value.
The numerical time grid determines the exercise opportunities used inside
each window. Refining that grid is part of a convergence study.

There is no `discount_index` on this contract. The context's fixed-income policy
maps USD to SOFR. This separates contractual terms from the market convention
used to discount them.

## Choose a model and a numerical method

Both configuration files assign a Hull–White model to SOFR, use mean reversion
`alpha = 0.05`, and calibrate its volatility schedule to ATM caplets sampled
from the constructed Black surface at 3M, 6M, 1Y, 2Y, 3Y and 5Y expiries.
Longer-expiry ATM forwards lie below the supplied smile's minimum strike;
the final calibrated sigma segment continues beyond 5Y for this six-year note.
For an interactive walkthrough, both use
48 time steps per year with a 4,096-step cap. The tree performs backward
induction on a trinomial lattice; the Fourier engine evolves values on a
256-point state grid spanning 10 standard deviations. These are two numerical
methods under the same calibrated model.

Time steps control both backward evolution and exercise opportunities inside
call windows. Fourier points control resolution across short-rate states.
Larger grids cost more time and memory, especially when differentiating value.
On this snapshot, each engine's price at these teaching settings differs by
less than 0.01 per 100 principal from its value at 96 time steps per year and
512 Fourier points. That comparison does not establish convergence: when
adapting the example, increase `steps_per_year` in both files and `points` in
the Fourier file until prices and required risk measures meet your tolerance.
Keep the state width (`standard_deviations`) under review as well.

The surface contains **Black quote volatilities**. Calibration turns option
prices into the model's absolute short-rate volatility schedule; no arbitrary
fixed sigma is substituted in the main valuation. Inspect or edit the linked
JSON files to change that choice.

In [ ]:
def price_with(
    configuration,
    anchor,
    requests,
):
    context = qs.PricingContext(
        quotes=market.quotes,
        curves=market.curves,
        csa_policy=market.csa_policy,
        volatility_surfaces=market.volatility_surfaces,
        pricer_configurations=configuration,
    )
    context.initialize()

    with context:
        return context.evaluate(
            qs.Trade(bond, ref, pricing_anchor=anchor),
            requests,
        )


# 75 bp, held fixed for both engines. Only the tree risk is used below.
anchor = qs.BondPricingAnchor.OAS(0.0075)
tree = price_with(
    tree_configuration,
    anchor,
    [qs.Request.Value, qs.Request.OAS, qs.Request.Sensitivities],
)
fourier = price_with(
    fourier_configuration,
    anchor,
    [qs.Request.Value, qs.Request.OAS],
)
comparison = pd.DataFrame(
    [
        {"engine": "Tree", "pv_usd": tree.price, "oas_bp": tree.oas * 10_000},
        {
            "engine": "Backward Fourier",
            "pv_usd": fourier.price,
            "oas_bp": fourier.oas * 10_000,
        },
    ]
).set_index("engine")
print(f"Absolute engine difference: USD {abs(tree.price - fourier.price):,.4f}")
comparison

Absolute engine difference: USD 26.2804


,pv_usd,oas_bp
engine,,
Tree,973988.335365,75.0
Backward Fourier,973962.054983,75.0


`price` is the signed USD position PV. The difference between engines reflects
their discretizations, not a bid/ask spread or a Monte Carlo standard error.
For a production tolerance, refine time and state grids and check stability;
one pair of grid settings is not proof of convergence. We request sensitivities
only for the tree result used in the risk report below. The Fourier comparison
and OAS inversion need only value and spread, avoiding unused risk calculations.

## Reverse the question: infer OAS from a price

An OAS anchor asks for price given a spread. A clean/dirty price anchor asks
for the spread that reproduces the observed price. Here we round-trip the tree
value into a dirty quote. Because the note starts on the valuation/settlement
date, its outstanding principal is the original notional and no accrued-interest
adjustment is needed. This simple conversion should not be copied unchanged
for a seasoned, amortized bond or a different settlement date.

In [19]:
dirty_quote = tree.price / 1_000_000 * 100
implied = price_with(
    tree_configuration,
    qs.BondPricingAnchor.dirty_price(dirty_quote),
    [qs.Request.Value, qs.Request.OAS],
)
pd.Series(
    {
        "dirty_price_per_100": dirty_quote,
        "implied_oas_bp": implied.oas * 10_000,
        "position_pv_usd": implied.price,
    }
)


dirty_price_per_100        97.398834
implied_oas_bp             75.000000
position_pv_usd        973988.335365
dtype: float64

## Read curve, volatility, and OAS risk separately

The sensitivity table contains original curve quotes, caplet quotes used in
calibration, and an independent `<identifier>.OAS` factor. For a price-anchored
valuation, the solved OAS is then held fixed when reporting curve and volatility
risk. Re-solving OAS after every shock would answer a different question.

Scale rate and OAS derivatives by `1e-4` for +1 bp, and Black-volatility
derivatives by `0.01` for +1 vol point. Unlike a straight bond, this callable
product has no yield-to-maturity or modified-duration request; its exercise
behavior is described by model value, OAS, and sensitivities.

In [20]:
risk = tree.sensitivities.set_index("pillar")["value"]
curve_labels = market.curves[0].to_dict()["quotes"]
vol_labels = market.volatility_surfaces[0].to_dict()["quotes"]
risk_report = pd.concat(
    [
        (
            risk.reindex(
                curve_labels,
                fill_value=0,
            )
            * 1e-4
        )
        .rename("pnl_usd")
        .to_frame()
        .assign(move="rate +1 bp"),
        (
            risk.reindex(
                vol_labels,
                fill_value=0,
            )
            * 0.01
        )
        .rename("pnl_usd")
        .to_frame()
        .assign(move="Black vol +1 point"),
        (risk.loc[[bond.identifier + ".OAS"]] * 1e-4)
        .rename("pnl_usd")
        .to_frame()
        .assign(move="OAS +1 bp"),
    ]
)
risk_report.loc[risk_report["pnl_usd"].abs().sort_values(ascending=False).index].head(
    12
)


,pnl_usd,move
pillar,,
USD-STEPPED-CALL-6Y.OAS,-328.362949,OAS +1 bp
CapletFloorlet_USD_SOFR_3M_1Y_Absolute_0.045_Straddle_Black,-81.180850,Black vol +1 point
CapletFloorlet_USD_SOFR_3M_2Y_Absolute_0.035_Straddle_Black,-79.799514,Black vol +1 point
CapletFloorlet_USD_SOFR_3M_3Y_Absolute_0.035_Straddle_Black,-60.409282,Black vol +1 point
CapletFloorlet_USD_SOFR_3M_6M_Absolute_0.045_Straddle_Black,-54.483488,Black vol +1 point
CapletFloorlet_USD_SOFR_3M_3M_Absolute_0.045_Straddle_Black,-53.481178,Black vol +1 point
OIS_USD_SOFR_1Y,-48.420103,rate +1 bp
CapletFloorlet_USD_SOFR_3M_1Y_Absolute_0.035_Straddle_Black,-37.773455,Black vol +1 point
CapletFloorlet_USD_SOFR_3M_6M_Absolute_0.055_Straddle_Black,-24.333361,Black vol +1 point


The companion [scripted-cashflow example](../scripting/scripted_cashflows.ipynb) values the
non-callable underlying on this calibrated market and retains a separate
zero-volatility representation check. Its script does not implement optimal
issuer exercise, so its value is not a second callable-bond price.